# Modeling 3: Serving Refactor

- 목적: `CustomerID` 누수를 제거하고, 서빙에 바로 연결할 모델과 스키마를 다시 확정한다.
- 비교 대상: Logistic Regression, Decision Tree, Random Forest, XGBoost, CatBoost
- 선택 기준: churn 확률을 직접 합산해 expected loss를 계산하므로, 분류 성능뿐 아니라 확률 품질(Brier / Log Loss)도 함께 본다.

In [1]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import TomekLinks
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, f1_score, log_loss, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
import pickle

PROJECT_DIR = Path.cwd().resolve().parents[0]
DATASET_PATH = PROJECT_DIR / 'datasets' / 'churn_preprocessed.csv'
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15
SELECTED_MODEL_NAME = 'xgboost'
EXPORTED_MODEL_PATH = ARTIFACT_DIR / 'model_xgb_v2_no_customer_id.pkl'
EXPORTED_SCHEMA_PATH = ARTIFACT_DIR / 'model_xgb_v2_no_customer_id_schema.json'
EXPORTED_METRICS_PATH = ARTIFACT_DIR / 'model_xgb_v2_no_customer_id_metrics.json'
BENCHMARK_PATH = ARTIFACT_DIR / 'benchmark_no_customer_id.csv'

In [2]:
df = pd.read_csv(DATASET_PATH)
df_model = df.drop(columns=['CustomerID'])
X = df_model.drop(columns=['Churn'])
y = df_model['Churn'].astype(int)

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()

feature_schema = [
    {
        'name': column,
        'dtype': 'numeric' if column in num_cols else 'categorical',
        'categories': [] if column in num_cols else sorted(map(str, X[column].dropna().unique().tolist())),
    }
    for column in X.columns
]

print('feature_count =', len(X.columns))
print('features =', list(X.columns))
print('categorical_columns =', cat_cols)
print('numeric_columns =', num_cols)

feature_count = 18
features = ['Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']
categorical_columns = ['PreferredLoginDevice', 'PreferredPaymentMode', 'Gender', 'PreferedOrderCat', 'MaritalStatus']
numeric_columns = ['Tenure', 'CityTier', 'WarehouseToHome', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'SatisfactionScore', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=y_train
)

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), cat_cols),
])

undersampler = TomekLinks(sampling_strategy='majority')

def build_models():
    return {
        'logistic_regression': ImbPipeline([
            ('preprocessor', preprocessor),
            ('undersampler', undersampler),
            ('classifier', LogisticRegression(max_iter=10000, class_weight='balanced', random_state=RANDOM_STATE)),
        ]),
        'decision_tree': ImbPipeline([
            ('preprocessor', preprocessor),
            ('undersampler', undersampler),
            ('classifier', DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced', max_depth=16, min_samples_split=2)),
        ]),
        'random_forest': Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=300, class_weight='balanced')),
        ]),
        'xgboost': ImbPipeline([
            ('preprocessor', preprocessor),
            ('undersampler', undersampler),
            ('classifier', XGBClassifier(random_state=RANDOM_STATE, learning_rate=0.5, n_estimators=30, eval_metric='logloss')),
        ]),
        'catboost': Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', CatBoostClassifier(random_state=RANDOM_STATE, verbose=0)),
        ]),
    }

In [4]:
def evaluate_candidate(name: str, model):
    model.fit(X_train, y_train)
    val_proba = model.predict_proba(X_val)[:, 1]
    thresholds = np.arange(0.2, 0.61, 0.05)

    best_threshold = 0.5
    best_f1 = -1.0
    for threshold in thresholds:
        pred = (val_proba >= threshold).astype(int)
        f1 = f1_score(y_val, pred)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = float(threshold)

    test_proba = model.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= best_threshold).astype(int)

    return {
        'model': name,
        'threshold': best_threshold,
        'val_pr_auc': round(float(average_precision_score(y_val, val_proba)), 6),
        'val_roc_auc': round(float(roc_auc_score(y_val, val_proba)), 6),
        'val_brier': round(float(brier_score_loss(y_val, val_proba)), 6),
        'val_log_loss': round(float(log_loss(y_val, val_proba)), 6),
        'test_pr_auc': round(float(average_precision_score(y_test, test_proba)), 6),
        'test_roc_auc': round(float(roc_auc_score(y_test, test_proba)), 6),
        'test_brier': round(float(brier_score_loss(y_test, test_proba)), 6),
        'test_log_loss': round(float(log_loss(y_test, test_proba)), 6),
        'test_precision': round(float(precision_score(y_test, test_pred)), 6),
        'test_recall': round(float(recall_score(y_test, test_pred)), 6),
        'test_f1': round(float(f1_score(y_test, test_pred)), 6),
    }

benchmark_results = [evaluate_candidate(name, model) for name, model in build_models().items()]
benchmark_df = pd.DataFrame(benchmark_results).sort_values(by=['test_pr_auc', 'test_roc_auc'], ascending=False)
benchmark_df.to_csv(BENCHMARK_PATH, index=False)
benchmark_df

,model,threshold,val_pr_auc,val_roc_auc,val_brier,val_log_loss,test_pr_auc,test_roc_auc,test_brier,test_log_loss,test_precision,test_recall,test_f1
2,random_forest,0.4,0.978279,0.994719,0.031828,0.129808,0.992955,0.998598,0.028257,0.123834,0.933333,0.985915,0.958904
3,xgboost,0.4,0.951486,0.987195,0.030460,0.107882,0.976872,0.993058,0.019884,0.080311,0.917808,0.943662,0.930556
4,catboost,0.4,0.954808,0.988911,0.033596,0.120167,0.973258,0.994490,0.027250,0.105522,0.902098,0.908451,0.905263
1,decision_tree,0.2,0.695770,0.905464,0.064067,2.309203,0.762558,0.923397,0.050760,1.720467,0.827815,0.880282,0.853242
0,logistic_regression,0.6,0.718016,0.901809,0.127438,0.393271,0.675272,0.893545,0.143580,0.439108,0.491379,0.802817,0.609626


## Model Selection Note

- `random_forest`가 ranking metric(PR AUC / ROC AUC / F1)은 가장 높게 나올 수 있다.
- 하지만 현재 제품 도메인은 threshold classification보다 `predict_proba`를 직접 합산해 expected loss를 계산하는 구조다.
- 그래서 확률 품질을 더 직접적으로 반영하는 `Brier score`와 `Log Loss`를 함께 본다. : https://rfriend.tistory.com/774 
- 이 기준에서는 `xgboost`가 가장 안정적인 probability estimator 후보로 선택된다.

이번 수정본은 `xgboost`를 서빙 후보로 export한다.

In [5]:
selected_row = benchmark_df.loc[benchmark_df['model'] == SELECTED_MODEL_NAME].iloc[0].to_dict()
selected_threshold = float(selected_row['threshold'])

X_train_full = pd.concat([X_train, X_val], axis=0)
y_train_full = pd.concat([y_train, y_val], axis=0)

selected_model = build_models()[SELECTED_MODEL_NAME]
selected_model.fit(X_train_full, y_train_full)

with EXPORTED_MODEL_PATH.open('wb') as file:
    pickle.dump(selected_model, file)

with EXPORTED_SCHEMA_PATH.open('w', encoding='utf-8') as file:
    json.dump(feature_schema, file, ensure_ascii=False, indent=2)

export_test_proba = selected_model.predict_proba(X_test)[:, 1]
export_test_pred = (export_test_proba >= selected_threshold).astype(int)
export_metrics = {
    'selected_model': SELECTED_MODEL_NAME,
    'threshold': selected_threshold,
    'test_pr_auc': round(float(average_precision_score(y_test, export_test_proba)), 6),
    'test_roc_auc': round(float(roc_auc_score(y_test, export_test_proba)), 6),
    'test_brier': round(float(brier_score_loss(y_test, export_test_proba)), 6),
    'test_log_loss': round(float(log_loss(y_test, export_test_proba)), 6),
    'test_precision': round(float(precision_score(y_test, export_test_pred)), 6),
    'test_recall': round(float(recall_score(y_test, export_test_pred)), 6),
    'test_f1': round(float(f1_score(y_test, export_test_pred)), 6),
}
with EXPORTED_METRICS_PATH.open('w', encoding='utf-8') as file:
    json.dump(export_metrics, file, ensure_ascii=False, indent=2)

print('exported_model =', EXPORTED_MODEL_PATH.name)
print('exported_schema =', EXPORTED_SCHEMA_PATH.name)
print('exported_metrics =', EXPORTED_METRICS_PATH.name)
export_metrics

exported_model = model_xgb_v2_no_customer_id.pkl
exported_schema = model_xgb_v2_no_customer_id_schema.json
exported_metrics = model_xgb_v2_no_customer_id_metrics.json


{'selected_model': 'xgboost',
 'threshold': 0.39999999999999997,
 'test_pr_auc': 0.989492,
 'test_roc_auc': 0.997896,
 'test_brier': 0.016312,
 'test_log_loss': 0.068389,
 'test_precision': 0.931973,
 'test_recall': 0.964789,
 'test_f1': 0.948097}

In [6]:
with EXPORTED_MODEL_PATH.open('rb') as file:
    loaded_model = pickle.load(file)

loaded_feature_names = list(loaded_model.feature_names_in_)
print('loaded_feature_count =', len(loaded_feature_names))
print('loaded_features =', loaded_feature_names)
assert 'CustomerID' not in loaded_feature_names
assert loaded_feature_names == list(X.columns)
print('schema_check = PASS')

loaded_feature_count = 18
loaded_features = ['Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']
schema_check = PASS
